In [1]:
import pandas as pd
import networkx as nx

# 1. Load the CSV into a DataFrame
# Replace 'tweets.csv' with the path to your input file
df = pd.read_csv('hashtags_15m.csv')

# 2. Compute an "hour" bucket for each timestamp (seconds since epoch // 3600)
df['hour'] = df['date_part'] // 3600

# 3. Map each unique hashtag string to a unique numeric ID
hashtag_ids = {tag: idx +1000000 for idx, tag in enumerate(df['hashtag'].unique(), start=1)}
df['hashtag_id'] = df['hashtag'].map(hashtag_ids)

# 4. Aggregate counts of (user_id, hashtag_id, hour)
agg = (
    df
    .groupby(['user_id', 'hashtag_id', 'hour'])
    .size()
    .reset_index(name='weight')
)


# 5. Build a bipartite graph in NetworkX
G = nx.Graph()
for _, row in agg.iterrows():
    user_node = f"user_{row['user_id']}"
    tag_node = f"hashtag_{row['hashtag_id']}"
    G.add_edge(
        user_node,
        tag_node,
        hour=int(row['hour']),
        weight=int(row['weight'])
    )

# 6. Export the aggregated edge list to a new CSV
# Columns: user, hashtag, hour, weight
agg.rename(columns={'user_id': 'user', 'hashtag_id': 'hashtag'}, inplace=True)
agg.to_csv('user_hashtag_hour_weight.csv', index=False)
agg.to_csv('../15m.csv', index=False)

print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")


Graph built with 100455 nodes and 368443 edges
